In [1]:
import re
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

## everef Market History File Update Exploration

Quick notebook to see how the market history files are updated relative to their market_date. Docs mention they are updated in place, so this is useful to see how frequently files are updated, and when their final update occurs.

In [2]:
BASE_URL = "https://data.everef.net/market-history/"

YEAR_RE = re.compile(r"^\d{4}/?$")
FILE_RE = re.compile(r"market-history-(\d{4}-\d{2}-\d{2})\.csv\.bz2$")
LAST_MODIFIED_RE = re.compile(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) UTC")

session = requests.Session()
session.headers.update({
    "User-Agent": "eve-market-history-lag-analysis/0.1"
})

In [3]:
resp = session.get(BASE_URL, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

year_urls = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(strip=True)

    if YEAR_RE.match(href) or YEAR_RE.match(text):
        year_urls.append(urljoin(BASE_URL, href))

year_urls = sorted(set(year_urls))

year_urls[:5], year_urls[-5:], len(year_urls)

In [4]:
test_year_url = year_urls[-1]
test_year_url

In [5]:
resp = session.get(test_year_url, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

rows = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    filename = href.rstrip("/").split("/")[-1]

    file_match = FILE_RE.match(filename)
    if not file_match:
        continue

    # Usually the useful metadata is in the parent table row.
    parent = a.find_parent("tr") or a.parent
    row_text = parent.get_text(" ", strip=True) if parent else a.get_text(" ", strip=True)

    modified_match = LAST_MODIFIED_RE.search(row_text)
    if not modified_match:
        print("Could not parse:", filename, row_text)
        continue

    market_date = file_match.group(1)
    modified_at = modified_match.group(1)

    rows.append({
        "market_date": market_date,
        "modified_at": modified_at,
        "filename": filename,
        "url": urljoin(test_year_url, href),
        "raw_text": row_text,
    })

test_df = pd.DataFrame(rows)
test_df.head()

In [6]:
all_rows = []

for year_url in tqdm(year_urls):
    resp = session.get(year_url, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        filename = href.rstrip("/").split("/")[-1]

        file_match = FILE_RE.match(filename)
        if not file_match:
            continue

        parent = a.find_parent("tr") or a.parent
        row_text = parent.get_text(" ", strip=True) if parent else a.get_text(" ", strip=True)

        modified_match = LAST_MODIFIED_RE.search(row_text)
        if not modified_match:
            print("Could not parse:", year_url, filename, row_text)
            continue

        all_rows.append({
            "market_date": file_match.group(1),
            "modified_at": modified_match.group(1),
            "filename": filename,
            "url": urljoin(year_url, href),
        })

len(all_rows)

In [7]:
df = pd.DataFrame(all_rows)

df["market_date"] = pd.to_datetime(df["market_date"], utc=True)
df["modified_at"] = pd.to_datetime(df["modified_at"], utc=True)

df["lag_timedelta"] = df["modified_at"] - df["market_date"]
df["lag_days_exact"] = df["lag_timedelta"].dt.total_seconds() / 86_400
df["lag_days_floor"] = df["lag_days_exact"].astype(int)

df = df.sort_values("market_date").reset_index(drop=True)

df.head()

In [8]:
df.tail()

In [9]:
df[["market_date", "modified_at", "lag_days_exact", "url"]].sample(10)

In [10]:
df["lag_days_exact"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [11]:
df.sort_values("lag_days_exact", ascending=False).head(20)[
    ["market_date", "modified_at", "lag_days_exact", "url"]
]

In [12]:
plt.figure(figsize=(12, 6))

plt.hist(df["lag_days_exact"], bins=100)

plt.title("EVE Ref market-history file modification lag")
plt.xlabel("Modified timestamp minus market date, in days")
plt.ylabel("Number of daily files")

plt.tight_layout()
plt.show()

In [13]:
plt.figure(figsize=(12, 6))

plt.hist(df["lag_days_exact"], bins=100)
plt.yscale("log")

plt.title("EVE Ref market-history file modification lag, log-count scale")
plt.xlabel("Modified timestamp minus market date, in days")
plt.ylabel("Number of daily files, log scale")

plt.tight_layout()
plt.show()

In [14]:
plt.figure(figsize=(14, 6))

plt.scatter(df["market_date"], df["lag_days_exact"], s=8)

plt.title("EVE Ref market-history modification lag by market date")
plt.xlabel("Market date")
plt.ylabel("Modified timestamp minus market date, in days")

plt.tight_layout()
plt.show()

In [16]:
recent_df = df[df["market_date"] >= "2023-01-01"].copy()

plt.figure(figsize=(14, 6))

plt.scatter(recent_df["market_date"], recent_df["lag_days_exact"], s=10)

plt.title("EVE Ref market-history modification lag, 2024+")
plt.xlabel("Market date")
plt.ylabel("Modified timestamp minus market date, in days")

plt.tight_layout()
plt.show()